# Provenance Density — minimal Colab demo

Runs a small (N≈10) audit on the same pipeline that produced the published numbers in Section 4.

**Where the work happens.** This notebook contains *no business logic*. The audit is split across these focused modules so a reader can verify each piece against the paper directly:

| File | Role | Paper section |
|---|---|---|
| `config.py` | All hyperparameters (β, λ, K, reputation tiers) | Table 1 |
| `source_scoring.py` | `w(s, c) = Reputation(s) · MatchRatio(σ, K_c)³` | Section 3.1 |
| `provenance_density.py` | `density(T) = tanh((1/β)·Σ_c (Σ_s w(s,c))^λ)` | Equation 1 |
| `semantic_entropy.py` | `P_int` via DeBERTa-v3-large NLI | Section 3.2 |
| `dataset.py` | TruthfulQA + FreshQA composite | Section 4 |
| `run_audit.py` | End-to-end driver | Section 4 |

Each module has a self-check at the bottom — run `python <module>.py` in a terminal to exercise its logic in isolation.

## 1. Install dependencies
Run this cell once. On Colab the package install takes ~2 minutes.

In [ ]:
!pip install -q -r code/requirements-lock.txt

## 2. Set API keys

**Do not hardcode keys in this notebook.** On Colab use the **Secrets** panel (key icon in the left sidebar) and add `OPENAI_API_KEY` and `SERPER_API_KEY`. The cell below pulls them from Colab secrets if available, otherwise from environment variables.

In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['SERPER_API_KEY'] = userdata.get('SERPER_API_KEY')
    print('Loaded keys from Colab secrets.')
except Exception:
    print('Not on Colab — relying on env vars OPENAI_API_KEY and SERPER_API_KEY.')

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY is not set.'
assert os.environ.get('SERPER_API_KEY'), 'SERPER_API_KEY is not set.'

## 3. Run a tiny audit

5 TruthfulQA + 5 FreshQA = 10 questions. End-to-end ~5 minutes on a T4 GPU, ~$0.05 in API costs.

To run the full N = 200 audit, change the arguments to `n_tqa=150, n_fresh=50` (or just call `run_audit()` with no args).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'code'))
from run_audit import run_audit
df = run_audit(n_tqa=5, n_fresh=5, output=Path('data/demo_results.csv'), seed=0)
df.head()

## 4. Reproduce the camera-ready figures from cached results

The next three cells regenerate Figures 2, 3, 4 of the paper from the cached audit log in `data/`. They make no API calls.

In [ ]:
!python code/roc_analysis.py            # Figure 2 + Table 2

In [ ]:
!python code/sensitivity_sweep.py       # Figure 3 + Table 3

In [ ]:
!python code/adversarial_probe.py       # Section 4.4 numbers
!python code/adversarial_figure.py      # Figure 4